# String Matching — Interactive Companion

Interactive visualizations for Chapter 1 of the Advanced Algorithms notes.  
Use the sliders and buttons to step through each algorithm on your own inputs.

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TheGhoul21/uniud-advanced-algorithms/HEAD?labpath=notebooks/01-string-matching.ipynb)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TheGhoul21/uniud-advanced-algorithms/blob/main/notebooks/01-string-matching.ipynb)

In [1]:
%config InlineBackend.figure_format = "retina"

import warnings
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np

warnings.filterwarnings("ignore", message=".*tight_layout.*")


def safe_tight_layout(fig=None):
    """Call tight_layout suppressing the aspect-ratio incompatibility warning."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        if fig is None:
            plt.tight_layout()
        else:
            fig.tight_layout()


plt.rcParams.update({
    "font.family": "monospace",
    "font.size": 13,
    "figure.facecolor": "#fafafa",
    "figure.dpi": 150,
    "savefig.dpi": 150,
})

COLORS = {
    "match": "#4CAF50",
    "mismatch": "#F44336",
    "current": "#FFC107",
    "skip": "#90CAF9",
    "default": "#E0E0E0",
    "pattern": "#BBDEFB",
    "border": "#7E57C2",
}


def draw_string_row(ax, y, string, label="", highlights=None, offset=0):
    """Draw a row of character boxes at vertical position y."""
    highlights = highlights or {}
    for i, ch in enumerate(string):
        color = highlights.get(i, COLORS["default"])
        rect = mpatches.FancyBboxPatch(
            (i + offset, y), 0.9, 0.9,
            boxstyle="round,pad=0.05",
            facecolor=color, edgecolor="#555", linewidth=1.2,
        )
        ax.add_patch(rect)
        ax.text(i + offset + 0.45, y + 0.45, ch,
                ha="center", va="center", fontsize=14, fontweight="bold")
    if label:
        ax.text(offset - 0.3, y + 0.45, label,
                ha="right", va="center", fontsize=12, color="#555")


def make_stepper(step_widget, label="Step"):
    """Create a prev/next/first/last button bar tied to an IntSlider (hidden)."""
    btn_first = widgets.Button(description="|<", layout=widgets.Layout(width="40px"))
    btn_prev  = widgets.Button(description="<", layout=widgets.Layout(width="40px"))
    btn_next  = widgets.Button(description=">", layout=widgets.Layout(width="40px"))
    btn_last  = widgets.Button(description=">|", layout=widgets.Layout(width="40px"))
    counter   = widgets.Label(value=f"{label}: {step_widget.value}/{step_widget.max}")

    def update_label(*_):
        counter.value = f"{label}: {step_widget.value}/{step_widget.max}"

    step_widget.observe(update_label, "value")
    step_widget.observe(update_label, "max")

    def on_first(_): step_widget.value = step_widget.min
    def on_prev(_):  step_widget.value = max(step_widget.min, step_widget.value - 1)
    def on_next(_):  step_widget.value = min(step_widget.max, step_widget.value + 1)
    def on_last(_):  step_widget.value = step_widget.max

    btn_first.on_click(on_first)
    btn_prev.on_click(on_prev)
    btn_next.on_click(on_next)
    btn_last.on_click(on_last)

    return widgets.HBox([btn_first, btn_prev, counter, btn_next, btn_last])


print("Helpers loaded (retina mode).")

Helpers loaded (retina mode).


---
## 1. Naive String Matching

Slide through every alignment of `P` against `T` and see which characters match.

In [2]:
def naive_trace(T, P):
    """Return list of (shift, comparisons) where each comparison is (j, matched)."""
    steps = []
    for s in range(len(T) - len(P) + 1):
        comps = []
        for j in range(len(P)):
            matched = T[s + j] == P[j]
            comps.append((j, matched))
            if not matched:
                break
        steps.append((s, comps))
    return steps


def draw_naive(T, P, step_idx):
    steps = naive_trace(T, P)
    n, m = len(T), len(P)
    s, comps = steps[step_idx]
    last_j, last_matched = comps[-1]
    is_full_match = last_matched and last_j == m - 1

    total_comps = sum(len(c) for _, c in steps[:step_idx + 1])
    matches_found = [st[0] for st in steps[:step_idx + 1]
                     if st[1][-1][1] and st[1][-1][0] == m - 1]

    fig, axes = plt.subplots(2, 1, figsize=(max(n * 0.85, 10), 5.5),
                              gridspec_kw={"height_ratios": [3, 2]})

    ax = axes[0]
    ax.set_xlim(-2.5, n + 1.5)
    ax.set_ylim(-1.0, 3.2)
    ax.set_aspect("equal")
    ax.axis("off")

    for i in range(n):
        ax.text(i + 0.45, 2.7, str(i), ha="center", va="center", fontsize=9, color="#999")

    t_hi = {}
    for j, matched in comps:
        t_hi[s + j] = COLORS["match"] if matched else COLORS["mismatch"]
    draw_string_row(ax, 1.6, T, label="T", highlights=t_hi)

    p_hi = {}
    for j, matched in comps:
        p_hi[j] = COLORS["match"] if matched else COLORS["mismatch"]
    draw_string_row(ax, 0.3, P, label="P", highlights=p_hi, offset=s)

    for j, matched in comps:
        color = COLORS["match"] if matched else COLORS["mismatch"]
        ax.plot([s + j + 0.45, s + j + 0.45], [1.6, 1.2], color=color, linewidth=1.5, alpha=0.5)

    status = "MATCH!" if is_full_match else f"mismatch: T[{s+last_j}]='{T[s+last_j]}' != P[{last_j}]='{P[last_j]}'"
    color = COLORS["match"] if is_full_match else COLORS["mismatch"]
    ax.text(n + 0.5, 1.0, status, fontsize=10, color=color, va="center", fontweight="bold")

    ax.set_title(f"Naive | shift s={s} | comparisons this step: {len(comps)} | "
                 f"total comparisons: {total_comps} | matches: {len(matches_found)}",
                 fontsize=11, pad=8)

    ax2 = axes[1]
    ax2.set_xlim(-0.5, len(steps))
    ax2.set_ylim(-0.5, 1.5)
    ax2.axis("off")

    for idx, (ss, cc) in enumerate(steps):
        last = cc[-1]
        if last[1] and last[0] == m - 1:
            c = COLORS["match"]
        elif idx <= step_idx:
            c = COLORS["mismatch"] if not last[1] else COLORS["default"]
        else:
            c = "#F5F5F5"
        edgecolor = "#000" if idx == step_idx else "#999"
        lw = 2.5 if idx == step_idx else 0.8
        rect = mpatches.FancyBboxPatch(
            (idx, 0.3), 0.8, 0.8, boxstyle="round,pad=0.03",
            facecolor=c, edgecolor=edgecolor, linewidth=lw)
        ax2.add_patch(rect)
        ax2.text(idx + 0.4, 0.7, str(len(cc)), ha="center", va="center", fontsize=8, fontweight="bold")
        ax2.text(idx + 0.4, 0.05, f"s={ss}", ha="center", va="center", fontsize=7, color="#777")

    ax2.set_title("All shifts (number = comparisons at each shift, current = bold border)",
                  fontsize=10, pad=5, loc="left")
    safe_tight_layout()
    plt.show()


T_input = widgets.Text(value="abcaabcabcaab", description="T:", layout=widgets.Layout(width="400px"))
P_input = widgets.Text(value="abcab", description="P:", layout=widgets.Layout(width="400px"))
step_slider = widgets.IntSlider(value=0, min=0, max=0, description="Step:", continuous_update=True)


def _update_naive_max(*_):
    T, P = T_input.value, P_input.value
    if T and P and len(P) <= len(T):
        step_slider.max = len(T) - len(P)

T_input.observe(_update_naive_max, "value")
P_input.observe(_update_naive_max, "value")
_update_naive_max()


def _draw_naive(T, P, step):
    if T and P and len(P) <= len(T):
        draw_naive(T, P, step)

out = widgets.interactive_output(_draw_naive, {"T": T_input, "P": P_input, "step": step_slider})
stepper = make_stepper(step_slider, "Shift")
display(T_input, P_input, stepper, out)

Text(value='abcaabcabcaab', description='T:', layout=Layout(width='400px'))

Text(value='abcab', description='P:', layout=Layout(width='400px'))

Output()

---
## 2. KMP — Failure Function (sp values) & Search

### sp (failure function) — Step by Step

Step through the computation of `sp[j]` = length of the longest proper border of `P[0..j]`. At each step you see:
- The **comparison** `P[k] vs P[j]`
- Any **fallbacks** through the sp chain
- The **prefix/suffix border** highlighted on the pattern

In [3]:
def compute_sp(P):
    """Compute sp[j] = length of longest proper border of P[0..j]."""
    m = len(P)
    sp = [0] * m
    k = 0
    for j in range(1, m):
        while k > 0 and P[k] != P[j]:
            k = sp[k - 1]
        if P[k] == P[j]:
            k += 1
        sp[j] = k
    return sp


def sp_trace(P):
    """Step-by-step sp computation. Each snapshot captures full state."""
    m = len(P)
    sp = [0] * m
    k = 0
    snapshots = []
    for j in range(1, m):
        fallbacks = []
        while k > 0 and P[k] != P[j]:
            old_k = k
            k = sp[k - 1]
            fallbacks.append((old_k, k))
        matched = P[k] == P[j]
        if matched:
            k += 1
        sp[j] = k
        snapshots.append({
            "j": j, "k_before": k - 1 if matched else k,
            "k_after": k, "matched": matched,
            "sp": list(sp), "fallbacks": fallbacks,
            "border_len": sp[j],
        })
    return snapshots


def draw_sp_step(P, step_idx):
    snapshots = sp_trace(P)
    if not snapshots:
        return
    step_idx = min(step_idx, len(snapshots) - 1)
    snap = snapshots[step_idx]
    m = len(P)
    j = snap["j"]
    sp = snap["sp"]
    border_len = snap["border_len"]
    matched = snap["matched"]
    k_cmp = snap["k_before"]

    fig, axes = plt.subplots(2, 1, figsize=(max(m * 0.85, 7), 5.0),
                              gridspec_kw={"height_ratios": [3, 2]})

    # ── Panel 1: Pattern with border highlights ──
    ax = axes[0]
    ax.set_xlim(-2.0, m + 1.0)
    ax.set_ylim(-1.8, 2.8)
    ax.set_aspect("equal")
    ax.axis("off")

    # P row highlights
    p_hi = {}
    if border_len > 0:
        for i in range(border_len):
            p_hi[i] = COLORS["match"]
        for i in range(j - border_len + 1, j + 1):
            p_hi[i] = COLORS["skip"]
    if matched:
        p_hi[k_cmp] = COLORS["match"]
        p_hi[j] = COLORS["match"]
    else:
        p_hi[j] = COLORS["mismatch"]

    draw_string_row(ax, 0.8, P, label="P", highlights=p_hi)

    # dim unprocessed positions
    for i in range(j + 1, m):
        rect = mpatches.FancyBboxPatch(
            (i, 0.8), 0.9, 0.9, boxstyle="round,pad=0.05",
            facecolor="#F0F0F0", edgecolor="#CCC", linewidth=0.8)
        ax.add_patch(rect)
        ax.text(i + 0.45, 1.25, P[i], ha="center", va="center",
                fontsize=14, fontweight="bold", color="#BBB")

    # index row above
    for i in range(m):
        ax.text(i + 0.45, 2.0, str(i), ha="center", fontsize=8, color="#999")

    # j pointer above the box
    ax.annotate(f"j={j}", xy=(j + 0.45, 1.7), xytext=(j + 0.45, 2.4),
                fontsize=9, ha="center", color="#C62828", fontweight="bold",
                arrowprops=dict(arrowstyle="->", color="#C62828", lw=1.5))

    # k pointer below the box
    ax.annotate(f"k={k_cmp}", xy=(k_cmp + 0.45, 0.8), xytext=(k_cmp + 0.45, 0.2),
                fontsize=9, ha="center", color="#1565C0", fontweight="bold",
                arrowprops=dict(arrowstyle="->", color="#1565C0", lw=1.5))

    # comparison text — placed to the right of the pattern, not between k and j
    cmp_color = COLORS["match"] if matched else COLORS["mismatch"]
    cmp_sym = "==" if matched else "!="
    cmp_text = f"P[{k_cmp}]='{P[k_cmp]}' {cmp_sym} P[{j}]='{P[j]}'"
    ax.text(m + 0.5, 1.25, cmp_text, fontsize=9, va="center", color=cmp_color, fontweight="bold")

    # border brackets below
    if border_len > 0:
        y_brk = -0.15
        # prefix
        ax.plot([0.1, border_len - 0.1], [y_brk, y_brk], color=COLORS["match"], lw=2.5, solid_capstyle="round")
        ax.text(border_len / 2, y_brk - 0.3, f"prefix[{border_len}]", ha="center",
                fontsize=8, color=COLORS["match"], fontweight="bold")
        # suffix
        suf_start = j - border_len + 1
        ax.plot([suf_start + 0.1, j + 0.8], [y_brk, y_brk], color=COLORS["skip"], lw=2.5, solid_capstyle="round")
        ax.text((suf_start + j + 1) / 2, y_brk - 0.3, f"suffix[{border_len}]", ha="center",
                fontsize=8, color="#1565C0", fontweight="bold")
        # border label
        ax.text(m / 2, -0.9, f"border = \"{P[:border_len]}\"  =>  sp[{j}] = {border_len}",
                ha="center", fontsize=10, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.2", facecolor="#E8F5E9", edgecolor=COLORS["match"]))
    else:
        ax.text(m / 2, -0.3, f"no border  =>  sp[{j}] = 0",
                ha="center", fontsize=10, color="#777")

    # fallback info
    if snap["fallbacks"]:
        fb_text = " -> ".join([f"k={old}->{new}" for old, new in snap["fallbacks"]])
        ax.text(m / 2, -1.3, f"fallbacks: {fb_text}", ha="center", fontsize=8,
                color=COLORS["border"],
                bbox=dict(boxstyle="round,pad=0.15", facecolor="#F3E5F5", edgecolor=COLORS["border"], alpha=0.8))

    ax.set_title(f"sp step {step_idx+1}/{len(snapshots)} | j={j} | sp[{j}]={sp[j]}", fontsize=10, pad=6)

    # ── Panel 2: sp array built so far ──
    ax2 = axes[1]
    ax2.set_xlim(-2.0, m + 0.5)
    ax2.set_ylim(-0.3, 2.2)
    ax2.set_aspect("equal")
    ax2.axis("off")

    for i in range(m):
        ax2.text(i + 0.45, 1.85, P[i], ha="center", fontsize=10, fontfamily="monospace",
                 color="#333" if i <= j else "#CCC")

    for i in range(m):
        computed = i <= j
        if i == j:
            color = COLORS["current"]
        elif computed and sp[i] > 0:
            color = "#E8D5F5"
        elif computed:
            color = "#FAFAFA"
        else:
            color = "#F0F0F0"
        rect = mpatches.FancyBboxPatch(
            (i, 0.3), 0.9, 0.9, boxstyle="round,pad=0.05",
            facecolor=color, edgecolor="#555" if computed else "#CCC",
            linewidth=1.0 if computed else 0.5)
        ax2.add_patch(rect)
        val = str(sp[i]) if computed else "?"
        ax2.text(i + 0.45, 0.75, val, ha="center", va="center", fontsize=12,
                 fontweight="bold" if computed else "normal",
                 color="#333" if computed else "#BBB")
        ax2.text(i + 0.45, 0.05, str(i), ha="center", fontsize=7, color="#999")
    ax2.text(-0.3, 0.75, "sp", ha="right", va="center", fontsize=11, color="#555")

    safe_tight_layout()
    plt.show()


P_sp_input = widgets.Text(value="abcabcab", description="P:", layout=widgets.Layout(width="400px"))
step_sp = widgets.IntSlider(value=0, min=0, max=0, description="Step:", layout=widgets.Layout(display="none"))


def _update_sp_max(*_):
    P = P_sp_input.value
    if P and len(P) >= 2:
        step_sp.max = max(len(sp_trace(P)) - 1, 0)
        step_sp.value = min(step_sp.value, step_sp.max)

P_sp_input.observe(_update_sp_max, "value")
_update_sp_max()


def _draw_sp(P, step):
    if P and len(P) >= 2:
        draw_sp_step(P, step)

out_sp = widgets.interactive_output(_draw_sp, {"P": P_sp_input, "step": step_sp})
stepper_sp = make_stepper(step_sp, "Step")
display(P_sp_input, stepper_sp, out_sp)

Text(value='abcabcab', description='P:', layout=Layout(width='400px'))

Output(outputs=({'output_type': 'display_data', 'data': {'text/plain': '<Figure size 1050x750 with 2 Axes>', '…

### KMP Search — Step by Step

Watch how KMP uses the failure function to avoid re-scanning characters. The arrow shows where `j` falls back to on a mismatch.

In [4]:
def kmp_trace(T, P):
    """Return list of snapshots with full state."""
    sp = compute_sp(P)
    n, m = len(T), len(P)
    snapshots = []
    j = 0
    total = 0
    matches = []
    for i in range(n):
        while j > 0 and T[i] != P[j]:
            total += 1
            old_j = j
            j = sp[j - 1]
            snapshots.append({
                "i": i, "j": old_j, "new_j": j, "event": "fallback",
                "align": i - old_j, "new_align": i - j,
                "total_comps": total, "matches": list(matches),
                "sp_used": old_j - 1,
            })
        total += 1
        if T[i] == P[j]:
            snapshots.append({
                "i": i, "j": j, "new_j": j, "event": "match_char",
                "align": i - j, "new_align": i - j,
                "total_comps": total, "matches": list(matches),
            })
            j += 1
            if j == m:
                matches.append(i - m + 1)
                snapshots.append({
                    "i": i, "j": j, "new_j": sp[j - 1], "event": "match_found",
                    "align": i - m + 1, "new_align": i - sp[j - 1] + 1,
                    "total_comps": total, "matches": list(matches),
                    "sp_used": j - 1,
                })
                j = sp[j - 1]
        else:
            snapshots.append({
                "i": i, "j": j, "new_j": j, "event": "mismatch",
                "align": i - j, "new_align": i - j,
                "total_comps": total, "matches": list(matches),
            })
    return snapshots


def draw_kmp_step(T, P, step_idx):
    sp = compute_sp(P)
    snapshots = kmp_trace(T, P)
    if not snapshots:
        return
    step_idx = min(step_idx, len(snapshots) - 1)
    snap = snapshots[step_idx]
    n, m = len(T), len(P)
    i, j, new_j = snap["i"], snap["j"], snap["new_j"]
    event = snap["event"]
    align = snap["align"]

    fig = plt.figure(figsize=(max(n * 0.85, 10), 8.5))
    gs = fig.add_gridspec(3, 1, height_ratios=[3.5, 2.5, 2.5], hspace=0.35)

    # ── Panel 1: T and P alignment ──
    ax = fig.add_subplot(gs[0])
    ax.set_xlim(-2.5, n + 2.0)
    ax.set_ylim(-1.2, 4.0)
    ax.set_aspect("equal")
    ax.axis("off")

    # index row
    for idx in range(n):
        ax.text(idx + 0.45, 3.5, str(idx), ha="center", fontsize=8, color="#999")

    # T row
    t_hi = {}
    if event == "match_char":
        t_hi[i] = COLORS["match"]
    elif event == "mismatch":
        t_hi[i] = COLORS["mismatch"]
    elif event == "fallback":
        t_hi[i] = COLORS["current"]
    elif event == "match_found":
        for k in range(m):
            t_hi[align + k] = COLORS["match"]
    draw_string_row(ax, 2.3, T, label="T", highlights=t_hi)

    # i pointer — above T row
    ax.annotate(f"i={i}", xy=(i + 0.45, 3.2), xytext=(i + 0.45, 3.7),
                fontsize=9, ha="center", color="#1565C0", fontweight="bold",
                arrowprops=dict(arrowstyle="->", color="#1565C0", lw=1.5))

    # P row
    p_hi = {}
    if event == "match_char":
        p_hi[j] = COLORS["match"]
    elif event == "mismatch":
        p_hi[j] = COLORS["mismatch"]
    elif event == "fallback":
        p_hi[j] = COLORS["mismatch"]
        if new_j < j:
            p_hi[new_j] = COLORS["current"]
    elif event == "match_found":
        for k in range(m):
            p_hi[k] = COLORS["match"]
    draw_string_row(ax, 0.7, P, label="P", highlights=p_hi, offset=align)

    # j pointer — below P row
    j_display = j if event != "match_found" else m - 1
    ax.annotate(f"j={j}", xy=(align + j_display + 0.45, 0.7),
                xytext=(align + j_display + 0.45, 0.1),
                fontsize=9, ha="center", color="#C62828", fontweight="bold",
                arrowprops=dict(arrowstyle="->", color="#C62828", lw=1.5))

    # fallback arrow — curved arc below the P row
    if event == "fallback" and new_j != j:
        ax.annotate("",
                     xy=(align + new_j + 0.45, 0.65),
                     xytext=(align + j + 0.45, 0.65),
                     arrowprops=dict(arrowstyle="-|>", color=COLORS["border"],
                                     lw=2.5, connectionstyle="arc3,rad=-0.4"))
        mid_x = align + (j + new_j) / 2 + 0.45
        ax.text(mid_x, -0.6,
                f"sp[{j-1}]={new_j}", ha="center", fontsize=10,
                color=COLORS["border"], fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.2", facecolor="white",
                          edgecolor=COLORS["border"], alpha=0.9))

    # event description
    if event == "fallback":
        info = f"FALLBACK: j={j} -> j=sp[{j-1}]={new_j} (T[{i}]='{T[i]}' != P[{j}]='{P[j]}')"
    elif event == "match_found":
        info = f"FULL MATCH at position {align}! Then j -> sp[{m-1}]={new_j}"
    elif event == "match_char":
        info = f"MATCH: T[{i}]=P[{j}]='{T[i]}', j -> {j+1}"
    else:
        info = f"MISMATCH: T[{i}]='{T[i]}' != P[{j}]='{P[j]}', j stays 0"

    ax.set_title(f"KMP step {step_idx+1}/{len(snapshots)} | comps={snap['total_comps']} | "
                 f"matches={snap['matches']}\n{info}", fontsize=10, pad=8)

    # ── Panel 2: sp table with highlight ──
    ax2 = fig.add_subplot(gs[1])
    ax2.set_xlim(-2.0, m + 0.5)
    ax2.set_ylim(-0.5, 2.5)
    ax2.set_aspect("equal")
    ax2.axis("off")
    ax2.set_title("Failure function sp[] (longest proper border of P[0..j])",
                  fontsize=10, pad=5, loc="left")

    for idx in range(m):
        ax2.text(idx + 0.45, 2.2, str(idx), ha="center", fontsize=8, color="#999")

    draw_string_row(ax2, 1.2, P, label="P")

    sp_used = snap.get("sp_used", None)
    for idx in range(m):
        if sp_used is not None and idx == sp_used:
            color = COLORS["current"]
        elif sp[idx] > 0:
            color = "#E8D5F5"
        else:
            color = "#F5F5F5"
        rect = mpatches.FancyBboxPatch(
            (idx, 0.0), 0.9, 0.9,
            boxstyle="round,pad=0.05",
            facecolor=color, edgecolor="#555", linewidth=1.0)
        ax2.add_patch(rect)
        ax2.text(idx + 0.45, 0.45, str(sp[idx]),
                 ha="center", va="center", fontsize=12, fontweight="bold")
    ax2.text(-0.3, 0.45, "sp", ha="right", va="center", fontsize=11, color="#555")

    if sp_used is not None and 0 <= sp_used < m and sp[sp_used] > 0:
        blen = sp[sp_used]
        ax2.plot([0.1, blen - 0.1], [-0.25, -0.25], color=COLORS["match"], lw=2.5)
        ax2.text(blen / 2, -0.45, "prefix", ha="center", fontsize=8, color=COLORS["match"])
        start = sp_used - blen + 2
        ax2.plot([start - 0.1, sp_used + 0.9], [-0.25, -0.25], color=COLORS["skip"], lw=2.5)
        ax2.text((start + sp_used + 1) / 2, -0.45, "border", ha="center", fontsize=8, color="#1565C0")

    # ── Panel 3: progress timeline ──
    ax3 = fig.add_subplot(gs[2])
    ax3.set_xlim(-0.5, len(snapshots))
    ax3.set_ylim(-0.3, 1.2)
    ax3.axis("off")
    ax3.set_title("Timeline (green=match, red=mismatch, yellow=fallback, star=full match)",
                  fontsize=9, pad=3, loc="left")

    event_colors = {
        "match_char": COLORS["match"],
        "mismatch": COLORS["mismatch"],
        "fallback": COLORS["current"],
        "match_found": "#2E7D32",
    }
    for idx, sn in enumerate(snapshots):
        c = event_colors.get(sn["event"], "#DDD")
        alpha = 1.0 if idx <= step_idx else 0.25
        marker = "*" if sn["event"] == "match_found" else "s"
        size = 80 if sn["event"] == "match_found" else 30
        ax3.scatter(idx, 0.5, c=c, s=size, marker=marker, alpha=alpha, edgecolors="#555", linewidth=0.5)
        if idx == step_idx:
            ax3.scatter(idx, 0.5, c="none", s=120, marker="o", edgecolors="#000", linewidth=2)

    safe_tight_layout(fig)
    plt.show()


# --- Widgets using interactive_output for reliable updates ---
T_kmp = widgets.Text(value="abcaabcabcaab", description="T:", layout=widgets.Layout(width="400px"))
P_kmp = widgets.Text(value="abcab", description="P:", layout=widgets.Layout(width="400px"))
step_kmp = widgets.IntSlider(value=0, min=0, max=1, description="Step:", continuous_update=True)


def _update_kmp_max(*_):
    """Keep slider max in sync with the number of KMP snapshots."""
    T, P = T_kmp.value, P_kmp.value
    if T and P and len(P) <= len(T):
        new_max = max(len(kmp_trace(T, P)) - 1, 0)
        step_kmp.max = new_max


T_kmp.observe(_update_kmp_max, "value")
P_kmp.observe(_update_kmp_max, "value")
_update_kmp_max()


def _draw_kmp(T, P, step):
    if T and P and len(P) <= len(T):
        draw_kmp_step(T, P, step)


out_kmp = widgets.interactive_output(_draw_kmp, {"T": T_kmp, "P": P_kmp, "step": step_kmp})
stepper_kmp = make_stepper(step_kmp, "Step")
display(T_kmp, P_kmp, stepper_kmp, out_kmp)

Text(value='abcaabcabcaab', description='T:', layout=Layout(width='400px'))

Text(value='abcab', description='P:', layout=Layout(width='400px'))

Output(outputs=({'output_type': 'display_data', 'data': {'text/plain': '<Figure size 1657.5x1275 with 3 Axes>'…

---
## 3. Z-Algorithm

Visualize the Z-array: `Z[k]` = length of the longest substring starting at `k` that matches a prefix of `S`.

In [5]:
def compute_z(S):
    """Compute Z-array for string S."""
    n = len(S)
    Z = [0] * n
    Z[0] = n
    l, r = 0, 0
    for k in range(1, n):
        if k < r:
            Z[k] = min(r - k, Z[k - l])
        while k + Z[k] < n and S[Z[k]] == S[k + Z[k]]:
            Z[k] += 1
        if k + Z[k] > r:
            l, r = k, k + Z[k]
    return Z


def z_trace(S):
    """Step-by-step Z computation returning full state at each k."""
    n = len(S)
    Z = [0] * n
    Z[0] = n
    snapshots = []
    l, r = 0, 0
    for k in range(1, n):
        init_from_zbox = False
        init_val = 0
        if k < r:
            init_val = min(r - k, Z[k - l])
            Z[k] = init_val
            init_from_zbox = True

        explicit_comps = 0
        while k + Z[k] < n and S[Z[k]] == S[k + Z[k]]:
            Z[k] += 1
            explicit_comps += 1

        old_l, old_r = l, r
        if k + Z[k] > r:
            l, r = k, k + Z[k]

        snapshots.append({
            "k": k, "Z": list(Z), "l": l, "r": r,
            "old_l": old_l, "old_r": old_r,
            "init_from_zbox": init_from_zbox,
            "init_val": init_val,
            "explicit_comps": explicit_comps,
            "zbox_updated": (l != old_l or r != old_r),
        })
    return snapshots


def draw_z_step(S, step_idx):
    snapshots = z_trace(S)
    if not snapshots:
        return
    step_idx = min(step_idx, len(snapshots) - 1)
    snap = snapshots[step_idx]
    n = len(S)
    k = snap["k"]
    Z = snap["Z"]
    l, r = snap["l"], snap["r"]
    zk = Z[k]

    fig, axes = plt.subplots(2, 1, figsize=(max(n * 0.75, 10), 5.5),
                              gridspec_kw={"height_ratios": [3, 2]})

    # ── Panel 1: String + Z-box + highlights ──
    ax = axes[0]
    ax.set_xlim(-2.0, n + 1.0)
    ax.set_ylim(-1.5, 3.2)
    ax.set_aspect("equal")
    ax.axis("off")

    # index row
    for i in range(n):
        ax.text(i + 0.45, 2.5, str(i), ha="center", fontsize=7, color="#999")

    # string row with highlights
    s_hi = {}
    if zk > 0:
        for j in range(zk):
            s_hi[j] = COLORS["match"]
            s_hi[k + j] = COLORS["skip"]
        s_hi[k] = COLORS["current"]
    else:
        s_hi[k] = COLORS["current"]
    draw_string_row(ax, 1.2, S, label="S", highlights=s_hi)

    # k pointer below
    ax.annotate(f"k={k}", xy=(k + 0.45, 1.2), xytext=(k + 0.45, 0.6),
                fontsize=9, ha="center", color="#C62828", fontweight="bold",
                arrowprops=dict(arrowstyle="->", color="#C62828", lw=1.5))

    # Z-box bracket above
    if r > 0:
        box_y = 2.8
        ax.plot([l, r - 0.1], [box_y, box_y], color=COLORS["border"], lw=2.5, solid_capstyle="round")
        ax.plot([l, l], [box_y - 0.1, box_y + 0.1], color=COLORS["border"], lw=1.5)
        ax.plot([r - 0.1, r - 0.1], [box_y - 0.1, box_y + 0.1], color=COLORS["border"], lw=1.5)
        ax.text((l + r) / 2, box_y + 0.2, f"Z-box [l={l}, r={r})",
                ha="center", fontsize=8, color=COLORS["border"], fontweight="bold")

    # brackets below for prefix and Z[k] match — on SEPARATE lines
    if zk > 0:
        # prefix bracket (higher line)
        y1 = -0.1
        ax.annotate("", xy=(0, y1), xytext=(zk - 0.1, y1),
                     arrowprops=dict(arrowstyle="<->", color=COLORS["match"], lw=1.5))
        ax.text(zk / 2, y1 - 0.35, f"prefix[0..{zk-1}]", ha="center",
                fontsize=8, color=COLORS["match"], fontweight="bold")

        # Z[k] bracket (lower line)
        y2 = -0.8
        ax.annotate("", xy=(k, y2), xytext=(k + zk - 0.1, y2),
                     arrowprops=dict(arrowstyle="<->", color="#1565C0", lw=2))
        ax.text(k + zk / 2, y2 - 0.35, f"Z[{k}]={zk}", ha="center",
                fontsize=9, color="#1565C0", fontweight="bold")

    # info text
    parts = []
    if snap["init_from_zbox"]:
        parts.append(f"Z-box reuse: min({snap['old_r']}-{k}, Z[{k-l}])={snap['init_val']}")
    if snap["explicit_comps"] > 0:
        parts.append(f"explicit comps: {snap['explicit_comps']}")
    if snap["zbox_updated"]:
        parts.append(f"Z-box: [{snap['old_l']},{snap['old_r']}) -> [{l},{r})")
    info = " | ".join(parts) if parts else f"Z[{k}]=0"

    ax.set_title(f"Z step {step_idx+1}/{len(snapshots)} | k={k} | Z[{k}]={zk}\n{info}",
                 fontsize=10, pad=6)

    # ── Panel 2: Full Z-array so far ──
    ax2 = axes[1]
    ax2.set_xlim(-2.0, n + 0.5)
    ax2.set_ylim(-0.3, 2.2)
    ax2.set_aspect("equal")
    ax2.axis("off")

    for i in range(n):
        computed = i == 0 or i <= k
        ax2.text(i + 0.45, 1.55, S[i], ha="center", fontsize=9, fontfamily="monospace",
                 color="#333" if computed else "#CCC")
        if i == k:
            color = COLORS["current"]
        elif computed and Z[i] > 0:
            color = "#E8F5E9"
        elif computed:
            color = "#FAFAFA"
        else:
            color = "#F0F0F0"
        rect = mpatches.FancyBboxPatch(
            (i, 0.3), 0.9, 0.9, boxstyle="round,pad=0.05",
            facecolor=color, edgecolor="#555" if computed else "#CCC",
            linewidth=1.0 if computed else 0.5)
        ax2.add_patch(rect)
        val = str(Z[i]) if computed else "?"
        ax2.text(i + 0.45, 0.75, val, ha="center", va="center", fontsize=10,
                 fontweight="bold" if computed else "normal",
                 color="#333" if computed else "#BBB")
        ax2.text(i + 0.45, 0.05, str(i), ha="center", fontsize=6, color="#999")
    ax2.text(-0.3, 0.75, "Z", ha="right", va="center", fontsize=11, color="#555")
    ax2.text(-0.3, 1.55, "S", ha="right", va="center", fontsize=11, color="#555")

    safe_tight_layout()
    plt.show()


S_z_input = widgets.Text(value="aabxaabxcaabxaabxay", description="S:", layout=widgets.Layout(width="500px"))
k_slider = widgets.IntSlider(value=0, min=0, max=0, description="Step:", layout=widgets.Layout(display="none"))


def _update_z_max(*_):
    S = S_z_input.value
    if S and len(S) >= 2:
        k_slider.max = max(len(z_trace(S)) - 1, 0)

S_z_input.observe(_update_z_max, "value")
_update_z_max()


def _draw_z(S, step):
    if S and len(S) >= 2:
        draw_z_step(S, step)

out_z = widgets.interactive_output(_draw_z, {"S": S_z_input, "step": k_slider})
stepper_z = make_stepper(k_slider, "Step")
display(S_z_input, stepper_z, out_z)

Text(value='aabxaabxcaabxaabxay', description='S:', layout=Layout(width='500px'))

Output(outputs=({'output_type': 'display_data', 'data': {'text/plain': '<Figure size 2137.5x825 with 2 Axes>',…

---
## 4. Boyer-Moore — Bad Character & Good Suffix

Step through the Boyer-Moore search. The algorithm scans the pattern **right-to-left** and uses two rules to skip alignments.

In [6]:
def bm_bad_char_table(P):
    """Last occurrence of each character in P (R function)."""
    R = {}
    for i, c in enumerate(P):
        R[c] = i
    return R


def bm_trace(T, P):
    """Boyer-Moore (bad character rule) returning detailed snapshots."""
    n, m = len(T), len(P)
    R = bm_bad_char_table(P)
    snapshots = []
    s = 0
    total_comps = 0
    matches = []
    while s <= n - m:
        comps = []
        j = m - 1
        while j >= 0 and P[j] == T[s + j]:
            comps.append((j, True))
            total_comps += 1
            j -= 1
        if j < 0:
            comps = [(k, True) for k in range(m - 1, -1, -1)]
            matches.append(s)
            snapshots.append({
                "s": s, "comps": comps, "event": "match_found", "shift": 1,
                "bad_char": None, "bad_pos": None, "j_mismatch": None,
                "total_comps": total_comps, "matches": list(matches),
            })
            s += 1
        else:
            total_comps += 1
            comps.append((j, False))
            bad_char = T[s + j]
            bad_pos = R.get(bad_char, -1)
            shift = max(1, j - bad_pos)
            snapshots.append({
                "s": s, "comps": comps, "event": "mismatch", "shift": shift,
                "bad_char": bad_char, "bad_pos": bad_pos, "j_mismatch": j,
                "total_comps": total_comps, "matches": list(matches),
            })
            s += shift
    return snapshots


def draw_bm_step(T, P, step_idx):
    R = bm_bad_char_table(P)
    snapshots = bm_trace(T, P)
    if not snapshots:
        return
    step_idx = min(step_idx, len(snapshots) - 1)
    snap = snapshots[step_idx]
    n, m = len(T), len(P)
    s = snap["s"]
    comps = snap["comps"]

    fig = plt.figure(figsize=(max(n * 0.85, 10), 8.0))
    gs = fig.add_gridspec(3, 1, height_ratios=[4, 2.5, 2], hspace=0.35)

    ax = fig.add_subplot(gs[0])
    ax.set_xlim(-2.5, n + 2.0)
    ax.set_ylim(-1.5, 3.5)
    ax.set_aspect("equal")
    ax.axis("off")

    for idx in range(n):
        ax.text(idx + 0.45, 3.0, str(idx), ha="center", fontsize=8, color="#999")

    t_hi = {}
    for j, matched in comps:
        t_hi[s + j] = COLORS["match"] if matched else COLORS["mismatch"]
    draw_string_row(ax, 1.8, T, label="T", highlights=t_hi)

    p_hi = {}
    for j, matched in comps:
        p_hi[j] = COLORS["match"] if matched else COLORS["mismatch"]
    draw_string_row(ax, 0.5, P, label="P", highlights=p_hi, offset=s)

    ax.annotate("", xy=(s + 0.3, 0.1), xytext=(s + m - 0.4, 0.1),
                arrowprops=dict(arrowstyle="<-", color="#999", lw=1.5, ls="--"))
    ax.text(s + m / 2, -0.25, "scan: right to left", ha="center", fontsize=8, color="#999")

    for order, (j, matched) in enumerate(comps):
        ax.text(s + j + 0.45, 0.5 - 0.15, str(order + 1), ha="center",
                fontsize=7, color="#999", style="italic")

    if snap["event"] == "mismatch":
        shift = snap["shift"]
        ax.annotate("", xy=(s + shift + m / 2, -0.7), xytext=(s + m / 2, -0.7),
                     arrowprops=dict(arrowstyle="->", color=COLORS["border"], lw=2.5))
        ax.text(s + m / 2 + shift / 2, -1.1, f"shift by {shift}",
                ha="center", fontsize=10, color=COLORS["border"], fontweight="bold")

    if snap["event"] == "match_found":
        info = f"FULL MATCH at position {s}"
    else:
        j_mm = snap["j_mismatch"]
        bc = snap["bad_char"]
        bp = snap["bad_pos"]
        info = (f"Mismatch at j={j_mm}: T[{s+j_mm}]='{bc}' != P[{j_mm}]='{P[j_mm]}' | "
                f"R('{bc}')={bp} | shift=max(1, {j_mm}-{bp})={snap['shift']}")

    ax.set_title(f"Boyer-Moore step {step_idx+1}/{len(snapshots)} | s={s} | "
                 f"comps={snap['total_comps']} | matches={snap['matches']}\n{info}",
                 fontsize=10, pad=8)

    ax2 = fig.add_subplot(gs[1])
    ax2.axis("off")
    ax2.set_title("Bad Character Table R(c) = rightmost position of c in P",
                  fontsize=10, pad=5, loc="left")

    all_chars = sorted(set(T + P))
    nc = len(all_chars)
    ax2.set_xlim(-0.5, nc + 0.5)
    ax2.set_ylim(-0.5, 2.5)
    ax2.set_aspect("equal")

    for idx, c in enumerate(all_chars):
        val = R.get(c, -1)
        if snap["event"] == "mismatch" and c == snap["bad_char"]:
            color = COLORS["mismatch"]
        elif val >= 0:
            color = "#E8F5E9"
        else:
            color = "#F5F5F5"
        rect = mpatches.FancyBboxPatch(
            (idx, 1.2), 0.9, 0.9, boxstyle="round,pad=0.05",
            facecolor="#FAFAFA", edgecolor="#555", linewidth=1.0)
        ax2.add_patch(rect)
        ax2.text(idx + 0.45, 1.65, f"'{c}'", ha="center", va="center",
                 fontsize=11, fontfamily="monospace")
        rect2 = mpatches.FancyBboxPatch(
            (idx, 0.0), 0.9, 0.9, boxstyle="round,pad=0.05",
            facecolor=color, edgecolor="#555", linewidth=1.0)
        ax2.add_patch(rect2)
        ax2.text(idx + 0.45, 0.45, str(val), ha="center", va="center",
                 fontsize=12, fontweight="bold")
    ax2.text(-0.3, 1.65, "c", ha="right", va="center", fontsize=11, color="#555")
    ax2.text(-0.3, 0.45, "R(c)", ha="right", va="center", fontsize=11, color="#555")

    ax3 = fig.add_subplot(gs[2])
    ax3.set_xlim(-0.5, len(snapshots))
    ax3.set_ylim(-0.5, 1.5)
    ax3.axis("off")
    ax3.set_title("Alignments tried (number = comparisons, green = match found)",
                  fontsize=9, pad=3, loc="left")

    for idx, sn in enumerate(snapshots):
        if sn["event"] == "match_found":
            color = COLORS["match"]
        elif idx <= step_idx:
            color = COLORS["mismatch"]
        else:
            color = "#F0F0F0"
        edgecolor = "#000" if idx == step_idx else "#999"
        lw = 2.5 if idx == step_idx else 0.8
        rect = mpatches.FancyBboxPatch(
            (idx, 0.3), 0.8, 0.8, boxstyle="round,pad=0.03",
            facecolor=color, edgecolor=edgecolor, linewidth=lw)
        ax3.add_patch(rect)
        ax3.text(idx + 0.4, 0.7, str(len(sn["comps"])),
                 ha="center", va="center", fontsize=8, fontweight="bold")
        ax3.text(idx + 0.4, 0.05, f"s={sn['s']}", ha="center", fontsize=7, color="#777")

    safe_tight_layout(fig)
    plt.show()


T_bm = widgets.Text(value="abcaabcabcaab", description="T:", layout=widgets.Layout(width="400px"))
P_bm = widgets.Text(value="abcab", description="P:", layout=widgets.Layout(width="400px"))
step_bm = widgets.IntSlider(value=0, min=0, max=0, description="Step:", continuous_update=True)


def _update_bm_max(*_):
    T, P = T_bm.value, P_bm.value
    if T and P and len(P) <= len(T):
        step_bm.max = max(len(bm_trace(T, P)) - 1, 0)

T_bm.observe(_update_bm_max, "value")
P_bm.observe(_update_bm_max, "value")
_update_bm_max()


def _draw_bm(T, P, step):
    if T and P and len(P) <= len(T):
        draw_bm_step(T, P, step)

out_bm = widgets.interactive_output(_draw_bm, {"T": T_bm, "P": P_bm, "step": step_bm})
stepper_bm = make_stepper(step_bm, "Step")
display(T_bm, P_bm, stepper_bm, out_bm)

Text(value='abcaabcabcaab', description='T:', layout=Layout(width='400px'))

Text(value='abcab', description='P:', layout=Layout(width='400px'))

Output()

---
## 5. Suffix Trie & Suffix Tree

Build and visualize the suffix trie and its compacted suffix tree for any input string. Uses `networkx` + `matplotlib` for tree layout.

In [7]:
import networkx as nx
from networkx.drawing.nx_agraph import graphviz_layout


def build_suffix_trie_graph(T):
    """Build a suffix trie as a networkx DiGraph. Keep it small (|T| <= 8)."""
    G = nx.DiGraph()
    G.add_node("root", label="root")
    node_id = 0
    for i in range(len(T)):
        cur = "root"
        for j in range(i, len(T)):
            ch = T[j]
            nxt = None
            for _, neighbor, data in G.out_edges(cur, data=True):
                if data["label"] == ch:
                    nxt = neighbor
                    break
            if nxt is None:
                node_id += 1
                name = f"n{node_id}"
                is_leaf = j == len(T) - 1
                G.add_node(name, label=str(i) if is_leaf else "")
                G.add_edge(cur, name, label=ch)
                cur = name
            else:
                cur = nxt
    return G


def build_suffix_tree_graph(T):
    """Build a compacted suffix tree as a networkx DiGraph."""
    # naive: build trie then compact
    class Node:
        _id = 0
        def __init__(self):
            Node._id += 1
            self.id = Node._id
            self.children = {}  # char -> (label_string, child_node)
            self.suffix_start = None

    Node._id = 0
    root = Node()
    for i in range(len(T)):
        cur = root
        j = i
        while j < len(T):
            ch = T[j]
            if ch in cur.children:
                label, child = cur.children[ch]
                # find how far we match along this edge
                k = 0
                while k < len(label) and j + k < len(T) and label[k] == T[j + k]:
                    k += 1
                if k == len(label):
                    cur = child
                    j += k
                else:
                    # split
                    mid = Node()
                    mid.children[label[k]] = (label[k:], child)
                    new_leaf = Node()
                    new_leaf.suffix_start = i
                    mid.children[T[j + k]] = (T[j + k:], new_leaf)
                    cur.children[ch] = (label[:k], mid)
                    break
            else:
                new_leaf = Node()
                new_leaf.suffix_start = i
                cur.children[ch] = (T[j:], new_leaf)
                break
        else:
            cur.suffix_start = i

    # convert to networkx
    G = nx.DiGraph()

    def add_nodes(node, name="root"):
        is_leaf = len(node.children) == 0
        lbl = str(node.suffix_start) if is_leaf and node.suffix_start is not None else ""
        G.add_node(name, label=lbl, is_leaf=is_leaf)
        for ch in sorted(node.children):
            edge_label, child = node.children[ch]
            child_name = f"n{child.id}"
            add_nodes(child, child_name)
            G.add_edge(name, child_name, label=edge_label)

    add_nodes(root)
    return G


def draw_tree_graph(G, title=""):
    """Draw a tree graph with edge labels."""
    if len(G.nodes) > 80:
        print("String too long for visualization (max ~8 chars).")
        return

    fig, ax = plt.subplots(figsize=(max(len(G.nodes) * 0.8, 6), max(len(G.nodes) * 0.5, 4)))

    try:
        pos = graphviz_layout(G, prog="dot")
    except Exception:
        pos = nx.spring_layout(G, seed=42)

    # separate leaf / internal
    leaves = [n for n in G.nodes if G.nodes[n].get("is_leaf", False) or G.out_degree(n) == 0]
    internals = [n for n in G.nodes if n not in leaves]

    nx.draw_networkx_nodes(G, pos, nodelist=internals, ax=ax,
                           node_color="#BBDEFB", node_size=350, edgecolors="#555")
    nx.draw_networkx_nodes(G, pos, nodelist=leaves, ax=ax,
                           node_color="#C8E6C9", node_size=350, edgecolors="#555",
                           linewidths=2.0)

    # node labels
    node_labels = {n: G.nodes[n].get("label", "") for n in G.nodes}
    nx.draw_networkx_labels(G, pos, labels=node_labels, ax=ax, font_size=10, font_weight="bold")

    nx.draw_networkx_edges(G, pos, ax=ax, edge_color="#777",
                           arrows=True, arrowsize=12, width=1.5)

    edge_labels = {(u, v): d["label"] for u, v, d in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, ax=ax,
                                  font_size=11, font_color="#9C27B0", font_family="monospace")

    ax.set_title(title, fontsize=13, pad=12)
    ax.axis("off")
    safe_tight_layout(fig)
    plt.show()


T_tree_input = widgets.Text(value="cacao$", description="T:", layout=widgets.Layout(width="300px"))
tree_type = widgets.ToggleButtons(options=["Suffix Trie", "Suffix Tree"], description="View:")
out_tree = widgets.Output()


def refresh_tree(_=None):
    T = T_tree_input.value
    if not T or len(T) > 10:
        with out_tree:
            clear_output(wait=True)
            print("Keep |T| <= 10 for readable visualization.")
        return
    with out_tree:
        clear_output(wait=True)
        if tree_type.value == "Suffix Trie":
            G = build_suffix_trie_graph(T)
            draw_tree_graph(G, title=f"Suffix Trie of \"{T}\"")
        else:
            G = build_suffix_tree_graph(T)
            draw_tree_graph(G, title=f"Suffix Tree of \"{T}\"")


T_tree_input.observe(refresh_tree, "value")
tree_type.observe(refresh_tree, "value")
display(T_tree_input, tree_type, out_tree)
refresh_tree()

Text(value='cacao$', description='T:', layout=Layout(width='300px'))

ToggleButtons(description='View:', options=('Suffix Trie', 'Suffix Tree'), value='Suffix Trie')

Output()

---
## 6. Algorithm Comparison

Run all algorithms on the same input and compare the number of character comparisons.

In [8]:
def count_naive(T, P):
    count = 0
    for s in range(len(T) - len(P) + 1):
        for j in range(len(P)):
            count += 1
            if T[s + j] != P[j]:
                break
    return count


def count_kmp(T, P):
    sp = compute_sp(P)
    count = 0
    j = 0
    for i in range(len(T)):
        while j > 0 and T[i] != P[j]:
            count += 1
            j = sp[j - 1]
        count += 1
        if T[i] == P[j]:
            j += 1
            if j == len(P):
                j = sp[j - 1]
    return count


def count_bm(T, P):
    n, m = len(T), len(P)
    R = bm_bad_char_table(P)
    count = 0
    s = 0
    while s <= n - m:
        j = m - 1
        while j >= 0:
            count += 1
            if P[j] != T[s + j]:
                break
            j -= 1
        if j < 0:
            s += 1
        else:
            bad = R.get(T[s + j], -1)
            s += max(1, j - bad)
    return count


def draw_comparison(T, P):
    naive_c = count_naive(T, P)
    kmp_c = count_kmp(T, P)
    bm_c = count_bm(T, P)

    algos = ["Naive", "KMP", "Boyer-Moore"]
    counts = [naive_c, kmp_c, bm_c]
    colors = ["#EF9A9A", "#90CAF9", "#A5D6A7"]

    fig, ax = plt.subplots(figsize=(7, 3.5))
    bars = ax.barh(algos, counts, color=colors, edgecolor="#555", height=0.55)
    for bar, c in zip(bars, counts):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                str(c), va="center", fontsize=13, fontweight="bold")

    ax.set_xlabel("Character comparisons", fontsize=12)
    ax.set_title(f"T = \"{T}\"   P = \"{P}\"   |T|={len(T)}  |P|={len(P)}", fontsize=12)
    ax.set_xlim(0, max(counts) * 1.2)
    safe_tight_layout(fig)
    plt.show()


T_cmp = widgets.Text(value="abcaabcabcaab", description="T:", layout=widgets.Layout(width="500px"))
P_cmp = widgets.Text(value="abcab", description="P:", layout=widgets.Layout(width="500px"))
out_cmp = widgets.Output()


def refresh_cmp(_=None):
    T, P = T_cmp.value, P_cmp.value
    if not T or not P or len(P) > len(T):
        return
    with out_cmp:
        clear_output(wait=True)
        draw_comparison(T, P)


T_cmp.observe(refresh_cmp, "value")
P_cmp.observe(refresh_cmp, "value")
display(T_cmp, P_cmp, out_cmp)
refresh_cmp()

Text(value='abcaabcabcaab', description='T:', layout=Layout(width='500px'))

Text(value='abcab', description='P:', layout=Layout(width='500px'))

Output()